# Cálculo del Cashback Perdido en el Mes Anterior

Este notebook filtra a los usuarios sin Hey Pro y calcula exactamente cuánto cashback dejaron de ganar en el último mes completo del dataset (`2025-11`), identificando la categoría donde perdieron más.

In [1]:
import pandas as pd
from pathlib import Path

# Rutas basadas en docs/CONTEXT.md asumiendo ejecución desde notebooks/uc3/
ROOT_DIR = Path().resolve().parent.parent
BASE_TXN = ROOT_DIR / "Datathon_Hey_2026_dataset_transacciones 1" / "dataset_transacciones"

print("Cargando clientes...")
df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})

print("Cargando transacciones...")
df_transacc = pd.read_csv(BASE_TXN / "hey_transacciones.csv",
                          dtype={"transaccion_id": str, "user_id": str, "producto_id": str},
                          parse_dates=["fecha_hora"])

Cargando clientes...
Cargando transacciones...


### Sanity Check: Validación con usuarios Pro
Se verifica que la regla `monto * 0.01` coincida con el `cashback_generado` existente.

In [2]:
pro_users = df_transacc[df_transacc["cashback_generado"].notna()].copy()
pro_users["cashback_calculado"] = pro_users["monto"] * 0.01
match_rate = (pro_users["cashback_calculado"].round(2) == pro_users["cashback_generado"].round(2)).mean()
assert match_rate > 0.95, f"Sanity check falló: {match_rate}"
print(f"Sanity check aprobado: {match_rate:.4f} de coincidencia.")

Sanity check aprobado: 0.9958 de coincidencia.


In [3]:
df_transacc['mes_anio'] = df_transacc['fecha_hora'].dt.to_period('M')
max_month = df_transacc['mes_anio'].max()
prev_month = max_month - 1
print(f"Mes máximo: {max_month}. Mes anterior utilizado para el cálculo: {prev_month}")

no_pro_users = df_clientes[df_clientes["es_hey_pro"] == False]["user_id"].unique()
print(f"Total de usuarios sin Hey Pro: {len(no_pro_users)}")

df_filtered = df_transacc[
    (df_transacc["user_id"].isin(no_pro_users)) &
    (df_transacc["tipo_operacion"] == "compra") &
    (df_transacc["estatus"] == "completada") &
    (df_transacc["mes_anio"] == prev_month)
].copy()

print(f"Transacciones a procesar del mes {prev_month}: {len(df_filtered)}")

Mes máximo: 2025-12. Mes anterior utilizado para el cálculo: 2025-11
Total de usuarios sin Hey Pro: 7680
Transacciones a procesar del mes 2025-11: 9446


In [4]:
df_filtered["cashback_potencial"] = df_filtered["monto"] * 0.01

# Total cashback por usuario
df_user_total = df_filtered.groupby("user_id")["cashback_potencial"].sum().reset_index()
df_user_total.rename(columns={"cashback_potencial": "cashback_perdido_mes"}, inplace=True)

# Top categoría por usuario
df_user_cat = df_filtered.groupby(["user_id", "categoria_mcc"])["cashback_potencial"].sum().reset_index()
df_user_cat = df_user_cat.sort_values(["user_id", "cashback_potencial"], ascending=[True, False])
df_top_cat = df_user_cat.drop_duplicates("user_id").copy()
df_top_cat.rename(columns={
    "categoria_mcc": "top_categoria_perdida",
    "cashback_potencial": "monto_top_categoria"
}, inplace=True)

df_results = df_user_total.merge(df_top_cat, on="user_id", how="left")

# Asegurarse de que incluimos a todos los usuarios sin Hey Pro
df_all_no_pro = pd.DataFrame({"user_id": no_pro_users})
df_final = df_all_no_pro.merge(df_results, on="user_id", how="left")
df_final["cashback_perdido_mes"] = df_final["cashback_perdido_mes"].fillna(0.0)
df_final["top_categoria_perdida"] = df_final["top_categoria_perdida"].fillna("ninguna")
df_final["monto_top_categoria"] = df_final["monto_top_categoria"].fillna(0.0)

df_final["cashback_perdido_mes"] = df_final["cashback_perdido_mes"].round(2)
df_final["monto_top_categoria"] = df_final["monto_top_categoria"].round(2)

output_file = ROOT_DIR / "outputs" / "uc3_cashback_perdido.csv"
output_file.parent.mkdir(exist_ok=True)
df_final.to_csv(output_file, index=False)
print(f"Resultados guardados en {output_file}")

display(df_final.head(10))

print("\n--- TOP 3 CATEGORÍAS DE PÉRDIDA AGREGADAS ---")
top_3_agregado = df_filtered.groupby("categoria_mcc")["cashback_potencial"].sum().sort_values(ascending=False).head(3)
for cat, monto in top_3_agregado.items():
    print(f"- {cat}: ${monto:,.2f} MXN perdidos en total")

Resultados guardados en C:\Users\nanoj\Documents\GitHub\Datathon-2026\outputs\uc3_cashback_perdido.csv


,user_id,cashback_perdido_mes,top_categoria_perdida,monto_top_categoria
0,USR-00004,23.38,restaurante,23.38
1,USR-00009,0.00,ninguna,0.00
2,USR-00010,25.09,servicios_digitales,15.19
3,USR-00011,9.22,viajes,9.22
4,USR-00013,5.37,supermercado,5.37
5,USR-00014,14.85,entretenimiento,14.85
6,USR-00015,0.00,ninguna,0.00
7,USR-00017,60.92,educacion,32.47
8,USR-00019,0.00,ninguna,0.00
9,USR-00023,13.10,salud,13.10



--- TOP 3 CATEGORÍAS DE PÉRDIDA AGREGADAS ---
- supermercado: $41,179.60 MXN perdidos en total
- restaurante: $27,476.82 MXN perdidos en total
- transporte: $23,418.88 MXN perdidos en total
